# Fase 2 - Preparacion de datos

Objetivo: explorar las tablas, entender el esquema relacional e integrar una base analitica confiable para el proyecto Bonsai Corp.

## 1. Configuración

Este notebook asume que se ejecuta con el kernel `Python (bonsai_env)`.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MPLCONFIGDIR = PROJECT_ROOT / ".matplotlib"
MPLCONFIGDIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

PROJECT_ROOT, DATA_DIR

(WindowsPath('e:/Bonsai-corps'), WindowsPath('e:/Bonsai-corps/data'))

## 2. Lectura de datos

Se leen inicialmente como texto para evitar conversiones prematuras y detectar problemas de formato.

In [2]:
files = {
    "catalogo_productos": DATA_DIR / "catalogo_productos.csv",
    "especificaciones_cajas": DATA_DIR / "especificaciones_cajas.csv",
    "operaciones_planta": DATA_DIR / "operaciones_planta.csv",
    "procurement_cajas": DATA_DIR / "procurement_cajas.csv",
}

raw = {
    name: pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8")
    for name, path in files.items()
}

{name: df.shape for name, df in raw.items()}

{'catalogo_productos': (435, 12),
 'especificaciones_cajas': (204, 16),
 'operaciones_planta': (435, 29),
 'procurement_cajas': (204, 19)}

## 3. Exploracion inicial de esquema

In [3]:
schema_summary = pd.DataFrame(
    [
        {
            "tabla": name,
            "filas": len(df),
            "columnas": df.shape[1],
            "columnas_lista": list(df.columns),
        }
        for name, df in raw.items()
    ]
)

schema_summary

,tabla,filas,columnas,columnas_lista
0,catalogo_productos,435,12,"[codigo_producto, descripcion_producto, ingred..."
1,especificaciones_cajas,204,16,"[caja_tipo_id, caja_grosor_mm, caja_interior_l..."
2,operaciones_planta,435,29,"[codigo_producto, volumen_producto_canal_servi..."
3,procurement_cajas,204,19,"[caja_tipo_id, volumen_tipo_planta_buenos_aire..."


## 4. Calidad de claves

Validar unicidad y duplicados antes de integrar tablas.

In [4]:
key_map = {
    "catalogo_productos": "codigo_producto",
    "operaciones_planta": "codigo_producto",
    "especificaciones_cajas": "caja_tipo_id",
    "procurement_cajas": "caja_tipo_id",
}

key_quality = []
for name, key in key_map.items():
    df = raw[name]
    key_quality.append(
        {
            "tabla": name,
            "clave": key,
            "filas": len(df),
            "valores_unicos": df[key].nunique(),
            "duplicados": df.duplicated(key).sum(),
            "claves_vacias": (df[key].str.strip() == "").sum(),
        }
    )

pd.DataFrame(key_quality)

,tabla,clave,filas,valores_unicos,duplicados,claves_vacias
0,catalogo_productos,codigo_producto,435,421,14,0
1,operaciones_planta,codigo_producto,435,421,14,0
2,especificaciones_cajas,caja_tipo_id,204,204,0,0
3,procurement_cajas,caja_tipo_id,204,204,0,0


In [5]:
def visualizar_duplicados(df, clave, nombre_tabla):
    dup = df[df.duplicated(clave, keep=False)].copy()
    dup = dup.sort_values(by=clave)

    resumen = (
        dup.groupby(clave)
        .size()
        .reset_index(name="repeticiones")
        .sort_values(["repeticiones", clave], ascending=[False, True])
    )

    print(f"Tabla: {nombre_tabla}")
    print(f"Registros duplicados: {len(dup)}")
    display(resumen)
    display(dup)

visualizar_duplicados(raw["catalogo_productos"], "codigo_producto", "catalogo_productos")
visualizar_duplicados(raw["operaciones_planta"], "codigo_producto", "operaciones_planta")

Tabla: catalogo_productos
Registros duplicados: 28


,codigo_producto,repeticiones
0,BR0092,2
1,BR0094,2
2,BR0136,2
3,BR0189,2
4,BR0193,2
5,BR0195,2
6,BR0209,2
7,BR0253,2
8,BR0310,2
9,BR0314,2


,codigo_producto,descripcion_producto,ingrediente_forma,tipo_proyecto,subcategoria,categoria,tamaño_corte,peso_neto_caja,tamaño_paquete,cantidad_paquetes,peso_neto_paquete,caja_tipo_id
91,BR0092,"5 bolsas de 2,5 kg | Bastones rectos de brocol...",Bastones rectos de brocoli,Estacional,Bastones clasicos - Reserva Privada,Bastones clasicos,Fino,12.5,"5 X2,5 KG",5,,4beade23001bd00ced8d86ccb4e4606b
92,BR0092,"5 bolsas de 2,5 kg | Bastones rectos de brocol...",Bastones rectos de brocoli,Estacional,Bastones clasicos - Reserva Privada,Bastones clasicos,Fino,12.5,"5 X 2,5 KG",5,2.5,0378f85c226113f4ac40fd360217bb8a
94,BR0094,BonsaiPremium Bastones rectos de brocoli 9x9 m...,Bastones rectos de brocoli,Estacional,Bastones clasicos - Reserva Privada,Bastones clasicos,Medio,12.5,"5 X 2,5 KG",5,2.5,c8c092bbf8da5f80452c66c38d27305f
95,BR0094,BonsaiPremium Bastones rectos de brocoli 9x9 m...,Bastones rectos de brocoli,Estacional,Bastones clasicos - Reserva Privada,Bastones clasicos,Medio,12.5,"5 X 2,5 KG",5,2.5,c8c092bbf8da5f80452c66c38d27305f
137,BR0136,Bastones rectos de brocoli 9x9 mm - clasico - ...,Bastones rectos de brocoli,Estacional,Bastones clasicos - Reserva Privada,Bastones clasicos,Medio,12.5,"5 X 2,5 KG",5,2.5,ecd82512cc8da13211c6468d7a34dc18
138,BR0136,Bastones rectos de brocoli 9x9 mm - clasico - ...,Bastones rectos de brocoli,Estacional,Bastones clasicos - Reserva Privada,Bastones clasicos,Medio,12.5,"5 X 2,5 KG",5,2.5,093e0a1389e0cc0f644ceea470a3cc01
191,BR0189,Corte rustico de brocoli - clasico - Caja de 5...,Corte rustico de brocoli,Forma estable,Bastones clasicos - Sigilo,Bastones clasicos,Forma,12.5,"5 X 2,5 KG",5,2.5,2bcfb295bee600aa6914bbd06f5891b6
192,BR0189,Corte rustico de brocoli - clasico - Caja de 5...,Corte rustico de brocoli,Forma estable,Bastones clasicos - Sigilo,Bastones clasicos,Forma,12.5,"5 X 2,5 KG",5,2.5,2bcfb295bee600aa6914bbd06f5891b6
196,BR0193,Brocoli zigzag 9x9 mm - clasico - Caja de 5 pa...,Brocoli zigzag,Forma estable,Bastones clasicos - Sigilo,Bastones clasicos,Forma,12.5,"5 X 2,5 KG",5,2.5,ca4da811f5581149b5b3f47ec84c9c5a
197,BR0193,Brocoli zigzag 9x9 mm - clasico - Caja de 5 pa...,Brocoli zigzag,Forma estable,Bastones clasicos - Sigilo,Bastones clasicos,Forma,12.5,5x2.5 kg,5,2.5,ca4da811f5581149b5b3f47ec84c9c5a


Tabla: operaciones_planta
Registros duplicados: 28


,codigo_producto,repeticiones
0,BR0092,2
1,BR0094,2
2,BR0136,2
3,BR0189,2
4,BR0193,2
5,BR0195,2
6,BR0209,2
7,BR0253,2
8,BR0310,2
9,BR0314,2


,codigo_producto,volumen_producto_canal_servicios_comida,volumen_producto_canal_minorista,volumen_producto_canal_cadenas_corporativas,volumen_producto_canal_otros,volumen_producto_total,volumen_producto_planta_buenos_aires,volumen_producto_planta_curitiba,volumen_producto_planta_santiago,volumen_producto_planta_monterrey,volumen_producto_planta_bakersfield,cantidad_pallets_planta_buenos_aires,cantidad_pallets_planta_curitiba,cantidad_pallets_planta_santiago,cantidad_pallets_planta_monterrey,cantidad_pallets_planta_bakersfield,cantidad_pallets_total,costo_total_planta_buenos_aires,costo_total_planta_curitiba,costo_total_planta_santiago,costo_total_planta_monterrey,costo_total_planta_bakersfield,costo_pallets_planta_buenos_aires,costo_pallets_planta_curitiba,costo_pallets_planta_santiago,costo_pallets_planta_monterrey,costo_pallets_planta_bakersfield,costo_pallets_total,costo_total
91,BR0092,994936,2,29,352,995319,0,367465,0,99701,528153,0,7656,0,2078,11004,20738,0.0,191081.8,0.0,45363.955,240309.615,0,1148400,0,311700,1650600,3110700,476755.37
92,BR0092,975733,2,29,345,976109,0,360373,0,97776,517960,0,6436,0,1746,9250,17432,0.0,187393.96,0.0,57198.96,235671.8,0,965400,0,261900,1387500,2614800,480264.72
94,BR0094,2580970,0,391,11548,2592909,0,789405,52684,1173993,576827,0,14097,941,20965,10301,46304,0.0,359179.275,27395.68,534166.8150000001,262456.28500000003,0,2114550,141150,3144750,1545150,6945600,1183198.0550000002
95,BR0094,2544037,0,385,11383,2555805,0,778109,51931,1157193,568572,0,13895,928,20665,10154,45642,0.0,354039.59500000003,27004.120000000003,526522.8150000001,258700.26,0,2084250,139200,3099750,1523100,6846300,1166266.79
137,BR0136,2,0,175960,0,175962,101927,74035,0,0,0,2124,1543,0,0,0,3667,48924.96,39978.9,0.0,0.0,0.0,318600,231450,0,0,0,550050,88903.86
138,BR0136,2,0,188628,0,188630,109265,79365,0,0,0,2277,1654,0,0,0,3931,52447.2,42857.100000000006,0.0,0.0,0.0,341550,248100,0,0,0,589650,95304.3
191,BR0189,528201,0,32401,0,560602,0,560602,0,0,0,0,10011,0,0,0,10011,0.0,255073.91,0.0,0.0,0.0,0,1501650,0,0,0,1501650,255073.91
192,BR0189,521786,0,32007,0,553793,0,553793,0,0,0,0,9890,0,0,0,9890,0.0,251975.815,0.0,0.0,0.0,0,1483500,0,0,0,1483500,251975.815
196,BR0193,352703,1,2,0,352706,0,0,264390,0,88316,0,0,5509,0,1840,7349,0.0,0.0,120297.45,0.0,45924.32,0,0,826350,0,276000,1102350,166221.77
197,BR0193,331408,1,2,0,331411,0,0,248427,0,82984,0,0,5176,0,1729,6905,0.0,0.0,113034.285,0.0,43151.68,0,0,776400,0,259350,1035750,156185.965


## 5. Proximos controles

- Normalizar columnas numericas.
- Crear `sku_line_id` para manejar duplicados de producto.
- Validar integridad relacional.
- Corregir volumenes de procurement.
- Imputar costos unitarios con `ERROR`.
- Derivar `cantidad_cajas_total` cuando este vacio.
- Construir dataset integrado a nivel SKU-linea-planta.